# 02 — Training Results

Visualize the SageMaker training job metrics:
- Training and validation loss curves
- LoRA trainable parameter count
- Training job summary from CloudWatch

In [ ]:
import boto3
import pandas as pd
import matplotlib.pyplot as plt

# ---- FILL THESE IN ----
JOB_NAME = 'arxiv-summarizer-finetune-YYYYMMDD-HHMMSS'  # your training job name
REGION   = 'us-east-1'
# -----------------------

sm = boto3.client('sagemaker', region_name=REGION)
job = sm.describe_training_job(TrainingJobName=JOB_NAME)
print(f"Status        : {job['TrainingJobStatus']}")
print(f"Instance      : {job['ResourceConfig']['InstanceType']}")
print(f"Training time : {job.get('TrainingTimeInSeconds', 'N/A')} seconds")
print(f"Billable time : {job.get('BillableTimeInSeconds', 'N/A')} seconds")

In [ ]:
# Pull loss metrics from CloudWatch
cw = boto3.client('cloudwatch', region_name=REGION)
from datetime import datetime, timedelta

def get_metric(metric_name, job_name):
    resp = cw.get_metric_statistics(
        Namespace='/aws/sagemaker/TrainingJobs',
        MetricName=metric_name,
        Dimensions=[{'Name': 'TrainingJobName', 'Value': job_name}],
        StartTime=datetime.utcnow() - timedelta(days=3),
        EndTime=datetime.utcnow(),
        Period=60,
        Statistics=['Average'],
    )
    points = sorted(resp['Datapoints'], key=lambda x: x['Timestamp'])
    return [p['Average'] for p in points]

train_loss = get_metric('train:loss', JOB_NAME)
eval_loss  = get_metric('eval:loss',  JOB_NAME)

plt.figure(figsize=(10, 4))
if train_loss:
    plt.plot(train_loss, label='Train Loss', color='steelblue')
if eval_loss:
    plt.plot(eval_loss,  label='Val Loss',   color='darkorange')
plt.title('Training and Validation Loss')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Final train loss: {train_loss[-1]:.4f}" if train_loss else 'No data yet')
print(f"Final val loss  : {eval_loss[-1]:.4f}"  if eval_loss  else 'No data yet')

In [ ]:
# LoRA parameter breakdown (run locally after training)
import torch
from peft import PeftModel
from transformers import AutoModelForSeq2SeqLM

MODEL_DIR = '../outputs/model'   # path to downloaded model

base = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
model = PeftModel.from_pretrained(base, MODEL_DIR)
model.print_trainable_parameters()

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal params    : {total:,}")
print(f"Trainable params: {trainable:,} ({trainable/total*100:.2f}%)")